# Python 进程死掉以后，Agent 还记得什么？

## V0.2 Persistence / Recovery

这个 lab 把“runtime object 死掉”和“durable semantic truth 仍在”分开看。

**Prediction / question:** 如果旧的 Python `Session` 对象被丢弃，新对象还能恢复多少已经发生的事实？

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside the AgentKernel repository.")

REPOSITORY_ROOT = find_repo_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
LABS_ROOT = REPOSITORY_ROOT / "examples" / "labs"
if str(LABS_ROOT) not in sys.path:
    sys.path.insert(0, str(LABS_ROOT))

from lab_helpers import event_rows, grant_rows, print_table, process_row, trajectory

## 1. Run once and persist Session events

In [ ]:
import asyncio
import json
import tempfile
from collections.abc import Mapping
from pathlib import Path

from agentkernel import (
    Agent, DefaultAgentLoop, JsonlSessionPersistence, MessageRole,
    ModelRequest, ModelResponse, PromptService, ScriptedLLM, Session,
    ToolCall, ToolDefinition, ToolExecutionContext, ToolRegistry, ToolSchema,
)
from agentkernel.protocol import JsonValue

async def add(arguments: Mapping[str, JsonValue], _context: ToolExecutionContext) -> JsonValue:
    return int(arguments["left"]) + int(arguments["right"])

def final_answer(request: ModelRequest) -> ModelResponse:
    payload = json.loads(request.messages[-1].content)
    return ModelResponse(content=f"final answer: {payload['output']}")

tmpdir = tempfile.TemporaryDirectory(prefix="agentkernel-lab-v0-2-")
path = Path(tmpdir.name) / "session.jsonl"
session = Session("lab-v0-2-session", JsonlSessionPersistence(path))
agent = Agent.create(agent_id="lab-v0-2-agent", session=session, capabilities={"math.add"})
tools = ToolRegistry()
tools.register(ToolDefinition(
    schema=ToolSchema("math.add", "Add two integers.", {"type": "object"}),
    handler=add,
    required_capability="math.add",
))
answer = asyncio.run(DefaultAgentLoop(
    llm=ScriptedLLM([
        ModelResponse(tool_calls=(ToolCall("call-add-1", "math.add", {"left": 7, "right": 35}),)),
        final_answer,
    ]),
    tools=tools,
    prompt=PromptService("Use the tool."),
).run(agent, "What is 7 + 35?"))
old_runtime_identity = id(session)
before_event_types = [event.type.value for event in session.events]
session.close()
print_table([
    {"fact": "answer before crash", "value": answer},
    {"fact": "old Session object id", "value": old_runtime_identity},
    {"fact": "persisted event count", "value": len(before_event_types)},
])

## 2. Discard old runtime object, load a fresh one

In [ ]:
restored = Session.load("lab-v0-2-session", JsonlSessionPersistence(path))
new_runtime_identity = id(restored)
restored_messages = restored.derive_messages()
print_table([
    {"fact": "old object id == new object id", "value": old_runtime_identity == new_runtime_identity},
    {"fact": "recovery status", "value": restored.recovery_analysis.status.value},
    {"fact": "restored events", "value": len(restored.events)},
    {"fact": "derived messages", "value": len(restored_messages)},
    {"fact": "durable facts lost", "value": before_event_types != [event.type.value for event in restored.events]},
])

## 3. Inspect replayed durable facts

In [ ]:
print_table(event_rows(restored))
trajectory(
    "Old Python objects discarded",
    "JSONL Session facts loaded",
    "Recovery analysis classifies Session",
    "Derived messages rebuilt from durable facts",
)
restored.close()
tmpdir.cleanup()

## Invariant

Runtime object identity is not durable truth. Session events are the replayable truth.

## WHAT THIS DEMONSTRATES / 本实验验证什么

- A fresh runtime object can recover persisted Session facts.
- Completed semantic events survive Python object loss.

## WHAT THIS DOES NOT DEMONSTRATE / 本实验不证明什么

- It does not test corrupted logs.
- It does not inject every crash prefix.
- It does not make in-memory objects durable.